### IMPORTS

In [1]:
import os
import random
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)

import evaluate
import librosa

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

C:\Users\edwin\OneDrive\Desktop\Convo AI\.gpuvenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Dataset Loading

In [2]:
train_df = pd.read_csv("hdsd_train.csv")
val_df = pd.read_csv("hdsd_val.csv")
test_df = pd.read_csv("hdsd_test.csv")

print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

(1633, 9)
(159, 9)
(210, 9)


Convert to HF Dataset

In [3]:
train_ds = Dataset.from_pandas(train_df)
val_ds = Dataset.from_pandas(val_df)
test_ds = Dataset.from_pandas(test_df)

In [4]:
print(train_ds.column_names)
print(train_ds[0])

['audio', 'text', 'speaker', 'utterance', 'mic', 'n_words', 'duration_sec', 'text_norm', 'text_devnagari']
{'audio': 'C:\\Users\\edwin\\OneDrive\\Desktop\\Capstone\\hindi indic\\HDSD\\hindi_sent\\CF02\\CF02_S1_H01_M2.wav', 'text': 'aapakei hindii pasanda karanei para khushii huii', 'speaker': 'CF02', 'utterance': 'CF02_S1_H01', 'mic': 'M2', 'n_words': 7, 'duration_sec': 3.069875, 'text_norm': 'aapake hindi pasanda karane para khushi hui', 'text_devnagari': 'आपके हिन्दि पसन्द करने पर खुशि हुइ'}


### Load Whisper Processor and Model

In [5]:
import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration

MODEL_NAME = "openai/whisper-small"

processor = WhisperProcessor.from_pretrained(
    MODEL_NAME,
    language="hi",
    task="transcribe"
)

model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
model.eval()

print(f"Using device: {device}")

Using device: cuda


In [6]:
import hashlib

def tensor_hash(t):
    return hashlib.sha256(
        t.detach().cpu().numpy().tobytes()
    ).hexdigest()

print("conv1      :", tensor_hash(model.model.encoder.conv1.weight))
print("conv2      :", tensor_hash(model.model.encoder.conv2.weight))
print("decoder emb:", tensor_hash(model.model.decoder.embed_tokens.weight))
print("lm_head    :", tensor_hash(model.proj_out.weight))

conv1      : cada141ad237828183f91ae14b8ce181a596f69b3af515d0d77792af8fa40590
conv2      : 241c1dde27999fb2c4f4003743d3ba772ba83d6b6072d89cfed9408887cc1016
decoder emb: aa35217c0e9866fbac49c90e19687a56a2f15c440182f976fdb82d63216593a3
lm_head    : aa35217c0e9866fbac49c90e19687a56a2f15c440182f976fdb82d63216593a3


In [7]:
import transformers

print("=" * 60)
print("INITIAL MODEL STATE")
print("=" * 60)

print("Transformers:", transformers.__version__)
print("Model class:", type(model))
print("Generation config id:", id(model.generation_config))
print("Config id:", id(model.config))
print("Config forced:", model.config.forced_decoder_ids)
print("Generation forced:", model.generation_config.forced_decoder_ids)

INITIAL MODEL STATE
Transformers: 4.46.3
Model class: <class 'transformers.models.whisper.modeling_whisper.WhisperForConditionalGeneration'>
Generation config id: 1486479228304
Config id: 1486479228400
Config forced: [[1, 50259], [2, 50359], [3, 50363]]
Generation forced: [[1, None], [2, 50359]]


In [8]:
print("Before assignment:")
print(model.generation_config.forced_decoder_ids)

model.generation_config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    language="hi",
    task="transcribe"
)

print("After assignment:")
print(model.generation_config.forced_decoder_ids)

Before assignment:
[[1, None], [2, 50359]]
After assignment:
[(1, 50276), (2, 50359), (3, 50363)]


In [9]:
print(model.generation_config.forced_decoder_ids)

[(1, 50276), (2, 50359), (3, 50363)]


In [10]:
sample = {
    "audio": r"C:\Users\edwin\OneDrive\Desktop\Capstone\hindi indic\HDSD\hindi_sent\CF00\CF00_S1_H01_M2.wav"
}

print(sample.keys())
print()


dict_keys(['audio'])



In [44]:
sample = test_ds[0]

audio, sr = librosa.load(sample["audio"], sr=16000)

inputs = processor(
    audio,
    sampling_rate=16000,
    return_tensors="pt"
)

input_features = inputs.input_features.to(device)

model.eval()
with torch.no_grad():
    pred_ids = model.generate(input_features)

print(processor.batch_decode(pred_ids, skip_special_tokens=True)[0])

 आपके हिंदी पसन्द कने पर खुषी हुई


In [11]:
import librosa
audio, sr = librosa.load(
    sample["audio"],
    sr=16000
)
print(audio.shape)
print(audio.dtype)
print(sr)

(55916,)
float32
16000


In [12]:
audio, sr = librosa.load(sample["audio"], sr=16000)

inputs = processor(
    audio,
    sampling_rate=16000,
    return_tensors="pt"
)

input_features = inputs.input_features.to(device)

with torch.no_grad():
    pred = model.generate(input_features)

print(processor.batch_decode(pred, skip_special_tokens=True)[0])

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


 आपके हिंदी पसन्द कने पर खुषी हुई


In [13]:
print(audio[:20])

[-0.00033569  0.00030518  0.00244141  0.00149536  0.00085449  0.00247192
  0.00198364  0.00100708  0.0020752   0.00149536  0.00204468  0.0022583
  0.00143433  0.00192261  0.0015564   0.00183105  0.00372314  0.00146484
  0.          0.00299072]


In [14]:
inputs = processor(
    audio,
    sampling_rate=16000,
    return_tensors="pt"
)

In [15]:
print(inputs.keys())

dict_keys(['input_features'])


In [16]:
print(inputs.input_features.shape)

torch.Size([1, 80, 3000])


The dimensions mean:

- 1 → batch size
- 80 → Whisper's log-Mel feature channels
- T → number of time frames

In [17]:
print(inputs.input_features.dtype)
print(inputs.input_features.device)

torch.float32
cpu


In [18]:
input_features = inputs.input_features.to(device)
print(input_features.device)

cuda:0


In [19]:
with torch.no_grad():
    predicted_ids = model.generate(
        input_features,
        language="hi",
        task="transcribe",
        use_cache=False,
    )

print(predicted_ids.shape)
print(
    processor.batch_decode(
        predicted_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]
)


You have passed task=transcribe, but also have set `forced_decoder_ids` to [(1, 50276), (2, 50359), (3, 50363)] which creates a conflict. `forced_decoder_ids` will be ignored in favor of task=transcribe.


torch.Size([1, 41])
 आपके हिंदी पसन्द कने पर खुषी हुई


In [20]:
inputs = processor(
    audio,
    sampling_rate=16000,
    return_tensors="pt"
)

features = inputs.input_features

print(features.shape)
print(features.dtype)

print(features.min().item())
print(features.max().item())
print(features.mean().item())
print(features.std().item())

print(features[0, 0, :10])
print(features[0, 20, :10])
print(features[0, 40, :10])
print(features[0, 79, :10])

torch.Size([1, 80, 3000])
torch.float32
-0.938666582107544
1.061333417892456
-0.8608372807502747
0.2607267200946808
tensor([ 0.1883,  0.1366, -0.0486,  0.0294,  0.0841, -0.1407, -0.1353, -0.0593,
        -0.0979, -0.2808])
tensor([-0.3424, -0.4981, -0.4668, -0.4001, -0.5072, -0.4776, -0.5107, -0.6582,
        -0.5475, -0.5643])
tensor([-0.6539, -0.6997, -0.4123, -0.3902, -0.5143, -0.4729, -0.4520, -0.2951,
        -0.3254, -0.3363])
tensor([-0.7823, -0.9387, -0.9387, -0.9387, -0.9387, -0.9387, -0.9387, -0.9387,
        -0.9387, -0.9387])


In [21]:
print(predicted_ids.shape)

torch.Size([1, 41])


In [23]:
model.generation_config.forced_decoder_ids = None

pred_ids = model.generate(
    features.to(model.device),
    language="hi",
    task="transcribe",
)

print(pred_ids.shape)
print(processor.batch_decode(pred_ids, skip_special_tokens=True))

You have passed task=transcribe, but also have set `forced_decoder_ids` to [[1, 50259], [2, 50359], [3, 50363]] which creates a conflict. `forced_decoder_ids` will be ignored in favor of task=transcribe.


torch.Size([1, 41])
[' आपके हिंदी पसन्द कने पर खुषी हुई']


In [24]:
#test a
model.generation_config.forced_decoder_ids = None

pred = model.generate(input_features)

print(pred.shape)
print(processor.batch_decode(pred, skip_special_tokens=True)[0])

torch.Size([1, 10])
 Your Hindi was very nice.


In [25]:
#test b
model.generation_config.forced_decoder_ids = None

pred = model.generate(
    input_features,
    language="hi",
    task="transcribe",
)

print(pred.shape)
print(processor.batch_decode(pred, skip_special_tokens=True)[0])

torch.Size([1, 41])
 आपके हिंदी पसन्द कने पर खुषी हुई


In [26]:
import sys
import torch
import transformers

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)

Python: C:\Users\edwin\OneDrive\Desktop\Convo AI\.gpuvenv\Scripts\python.exe
PyTorch: 2.5.1+cu121
Transformers: 4.46.3


In [27]:
#test c
model.generation_config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    language="hi",
    task="transcribe",
)

pred = model.generate(input_features)

print(pred.shape)
print(processor.batch_decode(pred, skip_special_tokens=True)[0])

torch.Size([1, 41])
 आपके हिंदी पसन्द कने पर खुषी हुई


In [28]:
print(transformers.__version__)

print(model.config.forced_decoder_ids)
print(model.generation_config.forced_decoder_ids)

print(processor.get_decoder_prompt_ids(
    language="hi",
    task="transcribe"
))

4.46.3
[[1, 50259], [2, 50359], [3, 50363]]
[(1, 50276), (2, 50359), (3, 50363)]
[(1, 50276), (2, 50359), (3, 50363)]


In [29]:
import torch

decoder_input_ids = torch.tensor(
    [[model.config.decoder_start_token_id]],
    device=model.device,
)

with torch.no_grad():
    outputs = model(
        input_features=input_features,
        decoder_input_ids=decoder_input_ids,
    )

logits = outputs.logits

print(logits.shape)

topk = torch.topk(logits[0, -1], k=20)

for score, idx in zip(topk.values, topk.indices):
    print(idx.item(), processor.tokenizer.decode([idx.item()]), score.item())

Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.43.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


torch.Size([1, 1, 51865])
50276 <|hi|> 23.511871337890625
50362 <|nocaptions|> 20.119220733642578
50259 <|en|> 19.124635696411133
50290 <|ur|> 18.865697860717773
50344 <|sa|> 17.242036819458008
50320 <|mr|> 15.96561336517334
50261 <|de|> 15.908585548400879
50321 <|pa|> 15.868372917175293
50287 <|ta|> 15.78165054321289
50299 <|te|> 15.663909912109375
50313 <|ne|> 15.124884605407715
50262 <|es|> 14.798049926757812
50289 <|th|> 14.592938423156738
50266 <|ja|> 14.442562103271484
50260 <|zh|> 14.43907356262207
50265 <|fr|> 14.411535263061523
50263 <|ru|> 14.328134536743164
50332 <|sd|> 14.323648452758789
50302 <|bn|> 14.30713939666748
50296 <|ml|> 14.178255081176758


In [30]:
print(pred_ids[0][:30].tolist())

[50258, 50276, 50359, 50363, 8485, 228, 3941, 103, 41858, 21981, 37139, 33279, 31945, 3941, 99, 31881, 8485, 103, 45938, 35082, 27099, 3941, 99, 31970, 35082, 21981, 8485, 103, 25411, 8485]


In [31]:
import transformers

print(transformers.__version__)
print(torch.__version__)
print(model.config._name_or_path)
print(processor.tokenizer.name_or_path)
print(processor.feature_extractor)

4.46.3
2.5.1+cu121
openai/whisper-small
openai/whisper-small
WhisperFeatureExtractor {
  "chunk_length": 30,
  "feature_extractor_type": "WhisperFeatureExtractor",
  "feature_size": 80,
  "hop_length": 160,
  "n_fft": 400,
  "n_samples": 480000,
  "nb_max_frames": 3000,
  "padding_side": "right",
  "padding_value": 0.0,
  "processor_class": "WhisperProcessor",
  "return_attention_mask": false,
  "sampling_rate": 16000
}



In [32]:
import transformers

print(transformers.__file__)
print(transformers.__version__)

C:\Users\edwin\OneDrive\Desktop\Convo AI\.gpuvenv\lib\site-packages\transformers\__init__.py
4.46.3


In [33]:
import pip

!pip show transformers

Name: transformers
Version: 4.46.3
Summary: State-of-the-art Machine Learning for JAX, PyTorch and TensorFlow
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/transformers/graphs/contributors)
Author-email: transformers@huggingface.co
License: Apache 2.0 License
Location: c:\users\edwin\appdata\local\programs\python\python310\lib\site-packages
Requires: filelock, huggingface-hub, numpy, packaging, pyyaml, regex, requests, safetensors, tokenizers, tqdm
Required-by: 


In [34]:
print(processor.get_decoder_prompt_ids(language="hi", task="transcribe"))

[(1, 50276), (2, 50359), (3, 50363)]


In [35]:
predicted_ids = model.generate(
    input_features,
    max_new_tokens=50
)

print(predicted_ids.shape)

print(
    processor.batch_decode(
        predicted_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )[0]
)

torch.Size([1, 41])
 आपके हिंदी पसन्द कने पर खुषी हुई


In [36]:
print(inputs.input_features.shape)
print(inputs.input_features.mean())
print(inputs.input_features.std())

torch.Size([1, 80, 3000])
tensor(-0.8608)
tensor(0.2607)


In [37]:
print(processor.tokenizer.name_or_path)

openai/whisper-small


In [38]:
print(processor.decode(predicted_ids[0]))
print(predicted_ids[0][:20])

<|startoftranscript|><|hi|><|transcribe|><|notimestamps|> आपके हिंदी पसन्द कने पर खुषी हुई
tensor([50258, 50276, 50359, 50363,  8485,   228,  3941,   103, 41858, 21981,
        37139, 33279, 31945,  3941,    99, 31881,  8485,   103, 45938, 35082],
       device='cuda:0')


In [39]:
from huggingface_hub import model_info

print(model.config._name_or_path)
print(model)

openai/whisper-small
WhisperForConditionalGeneration(
  (model): WhisperModel(
    (encoder): WhisperEncoder(
      (conv1): Conv1d(80, 768, kernel_size=(3,), stride=(1,), padding=(1,))
      (conv2): Conv1d(768, 768, kernel_size=(3,), stride=(2,), padding=(1,))
      (embed_positions): Embedding(1500, 768)
      (layers): ModuleList(
        (0-11): 12 x WhisperEncoderLayer(
          (self_attn): WhisperSdpaAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=False)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (fc2): Linear(in_features=3072, out_features=768

In [40]:
import sys
print(sys.executable)

C:\Users\edwin\OneDrive\Desktop\Convo AI\.gpuvenv\Scripts\python.exe


In [41]:
import sys
import transformers
import torch

print("Python:", sys.executable)
print("Transformers:", transformers.__version__)
print("Transformers location:", transformers.__file__)
print("Torch:", torch.__version__)

Python: C:\Users\edwin\OneDrive\Desktop\Convo AI\.gpuvenv\Scripts\python.exe
Transformers: 4.46.3
Transformers location: C:\Users\edwin\OneDrive\Desktop\Convo AI\.gpuvenv\lib\site-packages\transformers\__init__.py
Torch: 2.5.1+cu121


In [43]:
import transformers

print(transformers.__version__)
print(torch.__version__)
print(model.config._name_or_path)
print(processor.tokenizer.name_or_path)

4.46.3
2.5.1+cu121
openai/whisper-small
openai/whisper-small


### Preprocessing function

the preprocessing function is the bridge between our custom dataset and Whisper's expected input format.

Input {audio,text}

↓

Output {input_features, labels}